# Treino do modelo: Árvore de Regressão para preço de imóveis

**Objetivo:** estimar o preço de venda de um imóvel em São Paulo a partir da **área**, do **número de quartos** e do **bairro**.

**Dataset:** *São Paulo Real Estate - Sale / Rent - April 2019* (Kaggle). Dados reais de anúncios, não simulados.

Treinamos **dois modelos**, os dois com Árvore de Regressão e as mesmas entradas. Muda só **o que a árvore aprende a prever**:

| Modelo | O que a árvore prevê | Como vira preço |
|---|---|---|
| **Preço por m²** (principal, usado no site) | o valor de cada m² (ex.: R$ 8.000/m²) | multiplica pela área |
| **Preço total** (comparação) | o preço inteiro (ex.: R$ 640 mil) | já é o preço |

Etapas deste notebook:
1. Carregar e preparar os dados (funções do `preparacao.py`)
2. Montar o modelo: One-Hot Encoding do bairro + Árvore de Regressão
3. Escolher os parâmetros de cada modelo com validação cruzada
4. Medir o erro dos dois nos dados de teste
5. Conferir a comparação em 10 sorteios diferentes de treino/teste
6. Calcular a faixa de erro e a importância das variáveis
7. Salvar os dois modelos com joblib

## 1. Carregar e preparar os dados

A limpeza fica no `preparacao.py` para que o treino e o site usem exatamente as mesmas regras:
só vendas, **sem anúncios duplicados** (linhas idênticas em todas as colunas), sem ausentes, sem valores impossíveis e bairro sem o "/São Paulo".

Dois cuidados para o erro medido não ficar otimista:
- **Divisão por grupo:** imóveis com mesma área, quartos, bairro e preço (provavelmente unidades iguais do mesmo prédio) vão todos para o treino **ou** todos para o teste, nunca para os dois.
- **Extremos de preço por m²:** o 1% mais baixo e o 1% mais alto são removidos **depois** da divisão, com limites calculados só com o treino.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

from preparacao import (
    carregar_dados, preparar_dados, dividir_treino_teste,
    grupos_imovel,
)

df = carregar_dados()
vendas = df[df["Negotiation Type"] == "sale"]
dados = preparar_dados(df)
X_treino, X_teste, y_treino, y_teste, (preco_m2_min, preco_m2_max) = dividir_treino_teste(dados)

print("Vendas no dataset:", len(vendas))
print("Duplicatas removidas:", len(vendas) - len(vendas.drop_duplicates()))
print("Imóveis após a preparação:", len(dados))
print(f"Preço/m² mantido entre R$ {preco_m2_min:,.0f} e R$ {preco_m2_max:,.0f}")
print("Treino:", len(X_treino), "| Teste:", len(X_teste))

Vendas no dataset: 6412
Duplicatas removidas: 110
Imóveis após a preparação: 6302
Preço/m² mantido entre R$ 2,500 e R$ 17,889
Treino: 4951 | Teste: 1224


## 2. Montar o modelo

O modelo é um **Pipeline**, uma "linha de montagem" com duas etapas:

1. **ColumnTransformer:** transforma o bairro (texto) em colunas de 0 ou 1 com **One-Hot Encoding**. Área e quartos passam direto (`passthrough`).
   - `handle_unknown="ignore"`: se chegar um bairro que o modelo nunca viu, ele não quebra (todas as colunas de bairro ficam 0).
   - **Não normalizamos** área e quartos: a árvore divide os dados por limiares ("área > 85?"), e mudar a escala não muda as divisões.
2. **DecisionTreeRegressor:** a Árvore de Regressão. Ela faz perguntas sobre os dados até chegar a um grupo de imóveis parecidos e devolve a **média desse grupo**.

Os dois modelos usam exatamente este Pipeline. Com ele, o site manda `área, quartos, bairro` do jeito que o usuário digitou, e o próprio modelo faz a transformação.

In [2]:
def novo_modelo():
    """Pipeline usado pelos dois modelos: One-Hot do bairro + Árvore de Regressão."""
    transformador = ColumnTransformer(
        transformers=[
            ("bairro", OneHotEncoder(handle_unknown="ignore"), ["District"]),
        ],
        remainder="passthrough",  # Size e Rooms passam sem alteração
    )
    return Pipeline([
        ("preparo", transformador),
        ("arvore", DecisionTreeRegressor(random_state=42)),
    ])

novo_modelo()

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preparo', ...), ('arvore', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('bairro', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output 

### Os dois alvos: preço total e preço por m²

O preço total mistura duas coisas: o **tamanho** (imóvel maior custa mais em qualquer lugar) e a **localização** (o m² de Moema custa mais que o de Cidade Tiradentes).

- **Modelo preço total:** a árvore precisa descobrir sozinha que "maior custa mais" e gasta muitas perguntas nisso.
  Para um tamanho que não existe naquele bairro, ela devolve o preço do grupo mais parecido.
- **Modelo preço por m²:** a árvore prevê o valor do m² e o preço sai de `R$/m² × área`.
  A parte proporcional ao tamanho fica com a multiplicação, e as perguntas da árvore sobram para separar bairros.
  A área continua sendo entrada, então a árvore ainda pode dar R$/m² diferentes para imóveis pequenos e grandes.

O preço por m² **não é o mesmo para a cidade toda**: o bairro é entrada do modelo, então cada região tem o seu valor.

As duas funções abaixo fazem a conversão entre preço e alvo.

In [3]:
ALVOS = ["m2", "preco"]
NOME_ALVO = {"m2": "Preço por m²", "preco": "Preço total"}

def alvo_de(alvo, X, y):
    """O que a árvore aprende: o preço (alvo 'preco') ou o preço por m² (alvo 'm2')."""
    return y if alvo == "preco" else y / X["Size"]

def preco_previsto(alvo, modelo, X):
    """Previsão sempre em reais do preço final: no modelo por m², multiplica pela área."""
    previsto = modelo.predict(X)
    return previsto if alvo == "preco" else previsto * X["Size"].to_numpy()

## 3. Escolher os parâmetros (validação cruzada)

Uma árvore sem limite cresce até "decorar" o treino (**overfitting**) e erra muito em imóveis novos.
Uma árvore rasa demais é simples demais (**underfitting**). Dois parâmetros controlam isso:

- `max_depth`: quantas perguntas seguidas a árvore pode fazer.
- `min_samples_leaf`: quantos imóveis, no mínimo, cada grupo final precisa ter.

O **GridSearchCV** testa todas as combinações. Para cada uma, faz **validação cruzada com 5 partes**:
divide o treino em 5, treina em 4 e mede o erro na 5ª, repetindo 5 vezes. Usamos **GroupKFold**: imóveis do mesmo grupo ficam na mesma parte, como na divisão treino/teste.
**O teste não é usado aqui**, ele fica guardado para a avaliação final.

A métrica usada para escolher é o **MAE** (erro absoluto médio) **em reais do preço final**, nos dois modelos.
Assim a escolha é justa: o modelo por m² também é julgado pelo erro no preço, não no m².

In [4]:
PARAMETROS = {
    "arvore__max_depth": [3, 5, 8, 10, 12, 15, 20, None],
    "arvore__min_samples_leaf": [1, 5, 10, 20],
}

def pontuacao(alvo):
    """Pontuação do GridSearchCV: MAE em reais (negativo, porque o scikit-learn maximiza)."""
    def erro_em_reais(modelo, X, y_alvo):
        real = y_alvo if alvo == "preco" else y_alvo * X["Size"]
        return -mean_absolute_error(real, preco_previsto(alvo, modelo, X))
    return erro_em_reais

def treinar(alvo, X, y):
    """Escolhe os parâmetros por validação cruzada e devolve a busca já treinada."""
    busca = GridSearchCV(
        novo_modelo(), PARAMETROS,
        cv=GroupKFold(n_splits=5), scoring=pontuacao(alvo),
    )
    busca.fit(X, alvo_de(alvo, X, y), groups=grupos_imovel(X, y))
    return busca

buscas = {alvo: treinar(alvo, X_treino, y_treino) for alvo in ALVOS}

for alvo, busca in buscas.items():
    print(f"{NOME_ALVO[alvo]:<13} melhores parâmetros: {busca.best_params_}"
          f" | MAE na validação cruzada: R$ {-busca.best_score_:,.0f}")

Preço por m²  melhores parâmetros: {'arvore__max_depth': None, 'arvore__min_samples_leaf': 5} | MAE na validação cruzada: R$ 115,039
Preço total   melhores parâmetros: {'arvore__max_depth': None, 'arvore__min_samples_leaf': 1} | MAE na validação cruzada: R$ 127,707


In [5]:
# Como o erro do modelo principal muda com a profundidade (com o melhor min_samples_leaf)
busca = buscas["m2"]
resultados = pd.DataFrame(busca.cv_results_)
melhor_folha = busca.best_params_["arvore__min_samples_leaf"]
tabela = resultados[resultados["param_arvore__min_samples_leaf"] == melhor_folha]
tabela = tabela[["param_arvore__max_depth", "mean_test_score"]].copy()
tabela["MAE (R$)"] = (-tabela["mean_test_score"]).round(0)
tabela[["param_arvore__max_depth", "MAE (R$)"]].rename(columns={"param_arvore__max_depth": "max_depth"})

,max_depth,MAE (R$)
1,3,151449.0
5,5,143211.0
9,8,134944.0
13,10,130032.0
17,12,125965.0
21,15,120188.0
25,20,116622.0
29,None,115039.0


## 4. Avaliar nos dados de teste

Agora usamos os imóveis que **nenhum dos modelos viu**. Três métricas, todas sobre o preço final:

- **MAE:** erro médio em reais. Fácil de explicar: "em média, erra R$ X".
- **RMSE:** parecido com o MAE, mas pune mais os erros grandes.
- **R²:** fração da variação do preço que o modelo explica (1 = perfeito, 0 = não explica nada).

Também comparamos com um **modelo de referência** que sempre chuta a média de preço do treino.

In [6]:
modelos = {alvo: busca.best_estimator_ for alvo, busca in buscas.items()}
previstos = {alvo: preco_previsto(alvo, modelos[alvo], X_teste) for alvo in ALVOS}

referencia = DummyRegressor(strategy="mean").fit(X_treino, y_treino)

def metricas(real, prev):
    return {
        "MAE (R$)": round(mean_absolute_error(real, prev)),
        "RMSE (R$)": round(np.sqrt(mean_squared_error(real, prev))),
        "R²": round(r2_score(real, prev), 3),
    }

metricas_teste = {alvo: metricas(y_teste, previstos[alvo]) for alvo in ALVOS}
pd.DataFrame({
    **{NOME_ALVO[alvo]: metricas_teste[alvo] for alvo in ALVOS},
    "Referência (sempre a média)": metricas(y_teste, referencia.predict(X_teste)),
})

,Preço por m²,Preço total,Referência (sempre a média)
MAE (R$),108458.00,129418.000,346911.0
RMSE (R$),199452.00,297131.000,554149.0
R²,0.87,0.712,-0.0


## 5. A comparação se mantém em outros sorteios?

O resultado acima vem de **um único** sorteio de treino/teste, e um sorteio favorável pode enganar.
Repetimos tudo (divisão, escolha de parâmetros e teste) com **10 sementes diferentes**.
Isso demora um pouco, porque cada sorteio treina centenas de árvores.

In [7]:
linhas = []
for semente in range(10):
    Xa, Xb, ya, yb, _ = dividir_treino_teste(dados, semente=semente)
    linha = {"semente": semente}
    for alvo in ALVOS:
        modelo_s = treinar(alvo, Xa, ya).best_estimator_
        prev = preco_previsto(alvo, modelo_s, Xb)
        linha[f"MAE {NOME_ALVO[alvo]}"] = round(mean_absolute_error(yb, prev))
        linha[f"R² {NOME_ALVO[alvo]}"] = round(r2_score(yb, prev), 3)
    linhas.append(linha)

sorteios = pd.DataFrame(linhas).set_index("semente")
vitorias = int((sorteios["MAE Preço por m²"] < sorteios["MAE Preço total"]).sum())
reducao = 1 - sorteios["MAE Preço por m²"].mean() / sorteios["MAE Preço total"].mean()
print(f"Preço por m² teve MAE menor em {vitorias} de 10 sorteios.")
print(f"Redução média do erro: {reducao:.1%}")
sorteios

Preço por m² teve MAE menor em 10 de 10 sorteios.
Redução média do erro: 10.7%


,MAE Preço por m²,R² Preço por m²,MAE Preço total,R² Preço total
semente,,,,
0,111957,0.852,117637,0.802
1,118457,0.784,128504,0.796
2,119282,0.779,130144,0.771
3,112888,0.861,125889,0.786
4,106839,0.885,126257,0.782
5,115475,0.866,132976,0.751
6,114688,0.767,129735,0.690
7,109897,0.877,120269,0.822
8,102185,0.826,113247,0.758


In [8]:
sorteios.mean().round(3)

MAE Preço por m²    112390.300
R² Preço por m²          0.840
MAE Preço total     125844.500
R² Preço total           0.776
dtype: float64

**Conclusão:** o modelo por m² erra menos de forma consistente, não por sorte de um sorteio.
Por isso ele é o modelo principal do site. O modelo de preço total continua salvo, para comparação.

## 6. Faixa de erro e importância das variáveis

Um erro de R$ 100 mil é pequeno para um imóvel de R$ 3 milhões e enorme para um de R$ 200 mil.
Por isso a faixa do site é **percentual**: medimos o erro relativo de cada imóvel do teste
e pegamos o valor que cobre **80% dos casos**.

Assim, o site pode dizer: *"Em 80% dos imóveis de teste, o preço real ficou dentro desta faixa."*

In [9]:
faixas = {}
for alvo in ALVOS:
    erro_relativo = np.abs(y_teste.to_numpy() - previstos[alvo]) / previstos[alvo]
    faixas[alvo] = {
        "faixa_pct": float(np.quantile(erro_relativo, 0.80)),
        "erro_mediano_pct": float(np.median(erro_relativo)),
    }
    print(f"{NOME_ALVO[alvo]:<13} erro mediano: {faixas[alvo]['erro_mediano_pct']:.1%}"
          f" | faixa que cobre 80%: ±{faixas[alvo]['faixa_pct']:.1%}")

Preço por m²  erro mediano: 13.7% | faixa que cobre 80%: ±27.4%
Preço total   erro mediano: 12.8% | faixa que cobre 80%: ±29.6%


### Quais variáveis a árvore mais usou?

A **importância** mostra quanto cada variável ajudou a reduzir o erro nas divisões da árvore.
As 96 colunas de bairro são somadas em uma só, para comparar com área e quartos.
No modelo por m², o bairro deve pesar mais: o tamanho já é tratado pela multiplicação.

In [10]:
def importancias_agrupadas(modelo):
    nomes = modelo.named_steps["preparo"].get_feature_names_out()
    imp = pd.Series(modelo.named_steps["arvore"].feature_importances_, index=nomes)
    return {
        "Área": float(imp[[n for n in nomes if n.endswith("Size")]].sum()),
        "Quartos": float(imp[[n for n in nomes if n.endswith("Rooms")]].sum()),
        "Bairro": float(imp[[n for n in nomes if n.startswith("bairro__")]].sum()),
    }

importancias = {alvo: importancias_agrupadas(modelos[alvo]) for alvo in ALVOS}
pd.DataFrame({NOME_ALVO[a]: importancias[a] for a in ALVOS}).round(3)

,Preço por m²,Preço total
Área,0.370,0.823
Quartos,0.105,0.020
Bairro,0.526,0.157


### Teste com um imóvel de exemplo

In [11]:
exemplo = pd.DataFrame([{"Size": 80, "Rooms": 2, "District": "Pinheiros"}])
for alvo in ALVOS:
    preco = preco_previsto(alvo, modelos[alvo], exemplo)[0]
    faixa = faixas[alvo]["faixa_pct"]
    print(f"{NOME_ALVO[alvo]:<13} R$ {preco:,.0f}"
          f"  (faixa: R$ {preco * (1 - faixa):,.0f} a R$ {preco * (1 + faixa):,.0f})")
print(f"Preço por m² previsto: R$ {modelos['m2'].predict(exemplo)[0]:,.0f}/m²")

Preço por m²  R$ 760,143  (faixa: R$ 551,646 a R$ 968,640)
Preço total   R$ 694,500  (faixa: R$ 489,037 a R$ 899,963)
Preço por m² previsto: R$ 9,502/m²


## 7. Salvar os dois modelos

O site **não treina** a cada acesso: ele carrega os modelos prontos com `joblib`.
Cada modelo fica num arquivo separado:

- `modelo/modelo_arvore_m2.joblib`: **principal**, prevê o preço por m² (o site multiplica pela área).
- `modelo/modelo_arvore_preco.joblib`: **comparação**, prevê o preço total.

Junto, salvamos um arquivo de informações (`metadados.json`) com o que o site precisa para validar as entradas
e para a seção "Como funciona": lista de bairros, limites de área e quartos, e as métricas de cada modelo.

In [12]:
pasta_modelo = Path("modelo")
pasta_modelo.mkdir(exist_ok=True)

info_modelos = {}
for alvo in ALVOS:
    arquivo = f"modelo_arvore_{alvo}.joblib"
    joblib.dump(modelos[alvo], pasta_modelo / arquivo)
    info_modelos[alvo] = {
        "nome": NOME_ALVO[alvo],
        "arquivo": arquivo,
        "alvo": alvo,  # 'm2' = o site multiplica a previsão pela área
        "faixa_pct": round(faixas[alvo]["faixa_pct"], 4),
        "erro_mediano_pct": round(faixas[alvo]["erro_mediano_pct"], 4),
        "mae": metricas_teste[alvo]["MAE (R$)"],
        "rmse": metricas_teste[alvo]["RMSE (R$)"],
        "r2": metricas_teste[alvo]["R²"],
        "mae_10_sorteios": round(float(sorteios[f"MAE {NOME_ALVO[alvo]}"].mean())),
        "r2_10_sorteios": round(float(sorteios[f"R² {NOME_ALVO[alvo]}"].mean()), 3),
        "parametros": {k.replace("arvore__", ""): v for k, v in buscas[alvo].best_params_.items()},
        "importancias": {k: round(v, 3) for k, v in importancias[alvo].items()},
    }

metadados = {
    "modelo_principal": "m2",
    "modelos": info_modelos,
    "vitorias_m2_10_sorteios": vitorias,
    # limites e bairros vêm do treino: é o que o modelo realmente conhece
    "bairros": sorted(X_treino["District"].unique().tolist()),
    "area_min": int(X_treino["Size"].min()),
    "area_max": int(X_treino["Size"].max()),
    "quartos_min": int(X_treino["Rooms"].min()),
    "quartos_max": int(X_treino["Rooms"].max()),
    "preco_m2_min": round(preco_m2_min),
    "preco_m2_max": round(preco_m2_max),
    "linhas_treino": len(X_treino),
    "linhas_teste": len(X_teste),
    # faixa de área que o modelo viu em cada bairro: [mínimo, máximo, quantidade]
    "area_por_bairro": {
        bairro: [int(g["Size"].min()), int(g["Size"].max()), len(g)]
        for bairro, g in X_treino.groupby("District")
    },
    "linhas_dataset": len(df),
    "vendas_dataset": len(vendas),
    "duplicatas_removidas": len(vendas) - len(vendas.drop_duplicates()),
    "linhas_limpas": len(X_treino) + len(X_teste),
}
with open(pasta_modelo / "metadados.json", "w", encoding="utf-8") as f:
    json.dump(metadados, f, ensure_ascii=False, indent=2)

print("Modelos e metadados salvos em", pasta_modelo.resolve())

Modelos e metadados salvos em D:\mineração de dados projeto final\previsao-preco-imoveis\modelo
